# ДЗ №3. Ранжирование на основе datamart

Общая информация
Дата выдачи: 3 апреля 2026

Дедлайн: 26 апреля 2026 23:59 MSK

В этом домашнем задании мы продолжим строить приближенную к реальной рекомендательную систему. Работать будем с данными marketplace из [T-ECD](https://huggingface.co/datasets/t-tech/T-ECD).

Обычно рекомендательная система состоит из нескольких этапов:
1. Отбор кандидатов (Retrieval)
2. Ранжирование (Ranking)
3. Бизнес-логика (например, условие на то, чтобы товары от одного продавца не стояли в ленте друг за другом)

В этом домашнем задании сосредоточимся на втором этапе. Можно и нужно использовать наработки из предыдущего домашнего задания!

Краткое напоминание, почему отбор кандидатов и ранжирование - разные этапы. Задачу рекомендаций можно решать как регрессию (насколько релевантен айтем), классификацию (релевантен ли айтем) или ранжирование (какой из двух айтемов релевантнее). В идеале - проранжировать каталог под каждого пользователя. Но каталог всегда существенно больше того подмножества айтемов, которые пользователь увидит в итоговой выдаче. А качественно ранжировать весь каталог - ОЧЕНЬ долго и дорого. Получаем trade-off скорости и качества. Простое решение - многостадийные рекомендации. Сначала отберем кандидатов (релевантные/не релевантные), а потом проранжируем только релевантные.

В этом задании следующая разбалловка:

1) Cбор датамарта - 4 балла
2) Сбор датасета для обучения ранжирования - 2 балла
3) Сбор град. бустинга и оценка по метрикам с бейзлайном - 4 балла

Соответственно, максимум можно набрать 10 баллов.

In [3]:
!pip install -q polars lightgbm scikit-learn catboost shap optuna implicit torch

In [4]:
!pip install -U polars

In [5]:
import json
import gc
import os
import random
import typing as t
from abc import ABC, abstractmethod
from collections import defaultdict
from dataclasses import dataclass
from functools import partial
from pathlib import Path

import joblib
import lightgbm as lgbm
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import polars as pl
import polars.selectors as cs
import shap
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import HTML
from implicit.als import AlternatingLeastSquares
from optuna.samplers import TPESampler
from scipy.sparse import coo_matrix, csr_matrix
from tqdm.notebook import tqdm
from catboost import CatBoostRanker, Pool
from torch.utils.data import DataLoader, Dataset, IterableDataset

## Download Data

Данные занимают около 3.5 GB

In [6]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="t-tech/T-ECD",
    repo_type="dataset",
    local_dir=".",
    local_dir_use_symlinks=False,
    allow_patterns=["dataset/small/users.pq", "dataset/small/marketplace/**"]
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]

'/kaggle/working'

Больше всего места занимают эмбеддинги айтемов, их по желанию можно удалить, так как в этой работе нам потребуется дополнительное место на создание датамарта

In [7]:
pl.read_parquet("dataset/small/marketplace/items.pq").drop("embedding").write_parquet("dataset/small/marketplace/items.pq")

## I. Datamart (4 балла)

В задаче ранжирования хорошо себя показывают бустинги ([Catboost](https://catboost.ai), [LGBM](https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.Booster.html), [XGBoost](https://xgboost.readthedocs.io/en/stable/)). С точки зрения интерфейса: Algorithm(user, item, [features]), где фичи - любые полезные статистики (количество просмотров, конверсия из клика в кликаут, средний рейтинг, ...).

Каждый раз рассчитывать фичи с нуля по сырым логам достаточно тяжело (а в реальности рекомендации мы делаем не один раз в домашней работе, а гораздо чаще). Учитывая, что логи могут иметь разный формат или нуждаться в  дополнительной фильтрации (баги в логах всё же не редкость). Простое решение - предподсчитать статистики и сохранить их в отдельных файлах, которые затем удобно просто прочитать.

Это удобно сделать посредством датамарта. Датамарт - это витрина данных под определенную задачу. Он состоит из слоёв, где переход между слоями задает преобразование над данными, и обычно такие преобразования выполняются раз в какое-то время (в нашем случае пусть будет день). Для задачи ранжирования нам потребуется три слоя:
1. Raw - содержит сырые логи за каждый день. Мы уже собрали его на предыдущем шаге.
2. Aggs - содержит агрегированные статистики по юзерам и айтемам за каждый день (user, item, [stats]). Например, количество просмотров, количество кликов, количество кликов c поверхности поиска.
3. Features - содержит фичи, которые мы хотим использовать в модели, тоже за каждый день. Например, средний рейтинг айтема, средний рейтинг айтема по категориям, общая конверсия из клика в кликаут для пользователя за последние 30 дней. На этом слое удобно выделить отдельные папки по группам фичей (user, item, user-item, ...), чтобы избежать дубликатов при хранении.

Возьмем только небольшой срез данных, иначе дальнейшая работа может стать computationally infeasible. Переложим этот срез в `datamart/raw/events/{action_type}/{day}.pq`

Именно в таком формате логи обычно хранятся в сервисе.

Вы можете расширить условия на сэмплирование юзеров и айтемов. Если у вас будут проблемы с памятью, то можете и уменьшить что-то, но чем меньше ваш датасет, тем хуже будут метрики у конечной модели

In [8]:
ACTION_TYPES = ["view", "click", "clickout", "like"]
SUBDOMAINS = ["u2i", "i2i", "catalog", "search", "other"]

DAYS = list(range(1250, 1301))  # не все даты

Будем считать, что view < click < clickout < like с точки зрения бизнеса. Этот факт будет использоваться далее.

In [9]:
selected_users = (
    pl.concat(
        [pl.scan_parquet(f"dataset/small/marketplace/events/{str(day).zfill(5)}.pq") for day in DAYS[-10:]]
    )
    .group_by("user_id").agg(pl.len()).sort("len", descending=True)
    .head(20000).collect()["user_id"].to_list()
)

selected_items = (
    pl.concat(
        [pl.scan_parquet(f"dataset/small/marketplace/events/{str(day).zfill(5)}.pq") for day in DAYS[-10:]]
    ).group_by("item_id").agg(pl.len()).sort("len", descending=True)
    .head(20000).collect()["item_id"].to_list()
)

In [10]:
USERS = pl.scan_parquet("dataset/small/users.pq").filter(pl.col("user_id").is_in(selected_users)).collect()
print(USERS.shape)
ITEMS = pl.scan_parquet("dataset/small/marketplace/items.pq").filter(pl.col("item_id").is_in(selected_items)).collect()

(20000, 3)


### I.I. Datamart -> Raw слой (1 из 4 баллов)

Здесь вам надо собрать  raw слой:

![](images/datamart_raw.png)

в каждом файлике должны храниться данные на конкретную дату


In [11]:
BASE_RAW_PATH= Path("datamart/raw/events")
for action in ACTION_TYPES:
    (BASE_RAW_PATH / action).mkdir(parents=True, exist_ok=True)

for day in tqdm(DAYS, desc="Building Raw Layer"):
    day_str= str(day).zfill(5)
    source_path= f"dataset/small/marketplace/events/{day_str}.pq"

    if not os.path.exists(source_path):
        continue

    events_day = (
        pl.scan_parquet(source_path)
        .filter(
            (pl.col("user_id").is_in(selected_users)) &
            (pl.col("item_id").is_in(selected_items))
        )
    )

    df_filtered=events_day.collect()

    for action in ACTION_TYPES:
        action_df= df_filtered.filter(pl.col("action_type") == action)

        if not action_df.is_empty():
            target_path = BASE_RAW_PATH/action/ f"{day_str}.pq"
            action_df.write_parquet(target_path)

Building Raw Layer:   0%|          | 0/51 [00:00<?, ?it/s]

### I.II. Datamart -> Agg слой (1 из 4 баллов)

Рассчитате количество событий каждого типа (`action_type`) по каждой поверхности (`subdomain`) по парам (`user_id`, `item_id`) за каждый день. Не забудьте про общий счетчик - сумму по всем поверхностям. Сохраните в виде polars-таблиц.

Аналогично raw, но в agg значения внутри дня должны быть агрегированы

![](images/datamart_agg.png)

In [12]:
events_dir = Path("datamart/aggs/events/")
os.makedirs(events_dir, exist_ok=True)

for day in tqdm(DAYS, desc="Processing Agg layer"):
    day_str= str(day).zfill(5)
    action_dfs=[]

    for action in ACTION_TYPES:
        path= Path(f"datamart/raw/events/{action}/{day_str}.pq")

        if not path.exists():
            continue

        df= pl.read_parquet(path)

        agg_df= df.group_by(["user_id", "item_id"]).agg([
            pl.len().alias(f"num_{action}_all_subdomains"),
            *[
                pl.col("subdomain")
                .filter(pl.col("subdomain") == sub)
                .count()
                .alias(f"num_{action}_{sub}")
                for sub in SUBDOMAINS
            ]
        ])

        action_dfs.append(agg_df)

    if not action_dfs:
        continue

    res_df= action_dfs[0]
    for i in range(1, len(action_dfs)):
        res_df = res_df.join(
            action_dfs[i],
            on=["user_id","item_id"],
            how="full",
            coalesce=True  
        )

    res_df = res_df.fill_null(0)

    num_cols = [c for c in res_df.columns if c.startswith("num_")]
    res_df= res_df.with_columns([
        pl.col(num_cols).cast(pl.Float32)
    ])

    res_df.write_parquet(events_dir / f"{day_str}.pq")

Processing Agg layer:   0%|          | 0/51 [00:00<?, ?it/s]

Пример того, что может получиться.

In [13]:
pl.read_parquet("datamart/aggs/events/01300.pq").sample(10)

user_id,item_id,num_view_all_subdomains,num_view_u2i,num_view_i2i,num_view_catalog,num_view_search,num_view_other,num_click_all_subdomains,num_click_u2i,num_click_i2i,num_click_catalog,num_click_search,num_click_other,num_clickout_all_subdomains,num_clickout_u2i,num_clickout_i2i,num_clickout_catalog,num_clickout_search,num_clickout_other,num_like_all_subdomains,num_like_u2i,num_like_i2i,num_like_catalog,num_like_search,num_like_other
u64,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
78819414,"""nfmcg_2390489""",2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
85470983,"""nfmcg_5879908""",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
36705985,"""nfmcg_5879908""",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
32589217,"""nfmcg_22437058""",0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
53320705,"""nfmcg_27747063""",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
14472888,"""nfmcg_13962039""",1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
48074634,"""nfmcg_6002607""",0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
81001049,"""nfmcg_16293765""",1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
58623079,"""nfmcg_28013561""",0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### I.III. Datamart -> Feature слой (2 из 4 баллов)

Подумайте, какие признаки, рассчитанные на основе ранее собранных статистик, можно будет использовать в качестве фичей для модели. Реализуйте логику их подсчета за каждый день. Обратите внимание, что все фичи необходимо рассчитывать по какому-то временному окну, например "количество кликов на товар за последние 14 дней".

Сохраните признаки в виде polars-таблиц. Не забудьте про декомпозицию на отдельные папки по сущностям с целью избежать дубликатов при хранении.

Минимально должно получиться 10 признаков, из которых:
* 2 должны относиться только к сущности "пользователь" (например, медианная цена кликнутых айтемов у этого пользователя)
* 2 должны относиться только к сущности "айтем" (например, средняя конверсия из клика в кликаут по поверхности "поиск" у этого айтема)
* 6 признаков, которые показывают связь пользователя и айтема (например, количество просмотров этого айтема у этого пользователя).

Однако настоятельно рекомендуется собрать больше признаков. В данных много сущностей - категория, соцдем-кластер, бренд. И много параметров - цена, поверхность, тип события. В том числе можно рассчитать "изменение признака относительно предыдущего дня". Используйте polars expressions для написания шаблонного кода, в который затем удобно подставить конкретные названия сущностей и параметров, и получить готовый набор признаков.

P.S. Цену для товаров мы считаем фиксированной, она представлена в каталоге `ITEMS`

Может быть полезно посчитать конверсии как отношение числа второго события из пары к числу первого события из пары. Аккуратнее с делением на ноль - возможно, пользователей, для которых не определено число первого события из пары, не стоит учитывать при расчетах.

Пример того, как должно получиться:

![](images/datamart_feat2.png)

In [14]:
CONVERSION_PAIRS = [
    ("view", "click"),
    ("view", "clickout"),
    ("view", "like"),
    ("click", "clickout"),
    ("click", "like"),
    ("clickout", "like"),
]

Удобно называть фичи следующим образом.

In [15]:
def _feature_name(
    feature: str,
    keys: list[str],
    type: t.Literal["num", "cat"] = "num",
) -> str:
    clean_keys = [k for k in keys if k is not None]
    return f"f_{type}__{'_'.join(clean_keys)}___{feature}"
BASE_DIR= Path("datamart")
AGGS_DIR= BASE_DIR / "aggs" / "events"
FEATURES_DIR= BASE_DIR / "features"

In [16]:
def calculate_user_group_item_group_features(
    day_from: int = 1250,
    day_to: int = 1301,
    num_days: int = 14,
    user_group: t.Literal["region", "socdem_cluster"] | None = None,
    item_group: t.Literal["brand_id", "category", "subcategory", "item_id"] = "item_id"
) -> None:
    global USERS, ITEMS
    keys=[user_group, item_group]
    clean_keys = [k for k in keys if k is not None]
    save_dir= FEATURES_DIR / "-".join(clean_keys)
    save_dir.mkdir(parents=True, exist_ok=True)

    surfaces= ["u2i", "i2i", "catalog", "search", "other"]
    actions = ["view", "click", "clickout", "like"]
    
    cols_to_select = list(dict.fromkeys([k for k in ["item_id", item_group, "price"] if k]))
    i_m= ITEMS.select(cols_to_select)
    u_m= USERS.select(["user_id", user_group]) if user_group else None

    for target_day in tqdm(range(day_from, day_to), desc=f"UG-IG: {clean_keys}"):
        day_str = str(target_day).zfill(5)
        out_p = save_dir / f"{day_str}.pq"
        if out_p.exists(): out_p.unlink()
            
        daily_dfs = []
        for d in range(target_day - num_days, target_day):
            p = AGGS_DIR / f"{str(d).zfill(5)}.pq"
            if p.exists():
                df = pl.scan_parquet(str(p)).join(i_m.lazy(), on="item_id")
                if u_m is not None: df = df.join(u_m.lazy(), on="user_id")
                
                aggs = []
                for a in actions:
                    aggs.append(pl.col(f"num_{a}_all_subdomains").sum().alias(f"cnt_{a}"))
                    for s in surfaces:
                        aggs.append(pl.col(f"num_{a}_{s}").sum().alias(f"cnt_{a}_{s}"))
                    aggs.append(pl.col("price").filter(pl.col(f"num_{a}_all_subdomains") > 0).alias(f"p_{a}"))
                
                daily_dfs.append(df.group_by(clean_keys).agg(aggs).collect())

        if not daily_dfs: continue
        combined = pl.concat(daily_dfs).group_by(clean_keys).agg([
            *[pl.col(f"cnt_{a}").sum().alias(_feature_name(f"num_{a}_{num_days}d", clean_keys)) for a in actions],
            *[pl.col(f"cnt_{a}_{s}").sum().alias(_feature_name(f"num_{a}_from_{s}_{num_days}d", clean_keys)) for a in actions for s in surfaces],
            *[pl.col(f"p_{a}").list.explode().median().alias(_feature_name(f"median_price_{a}_{num_days}d", clean_keys)) for a in actions]
        ])

        for p1, p2 in CONVERSION_PAIRS:
            n1, n2= _feature_name(f"num_{p1}_{num_days}d", clean_keys), _feature_name(f"num_{p2}_{num_days}d", clean_keys)
            combined = combined.with_columns((pl.col(n2)/pl.col(n1).replace(0,None)).fill_nan(0).alias(_feature_name(f"conv_{p1}_{p2}", clean_keys)))
            for s in surfaces:
                ns1, ns2= _feature_name(f"num_{p1}_from_{s}_{num_days}d", clean_keys), _feature_name(f"num_{p2}_from_{s}_{num_days}d", clean_keys)
                combined= combined.with_columns((pl.col(ns2)/pl.col(ns1).replace(0,None)).fill_nan(0).alias(_feature_name(f"conv_{p1}_{p2}_{s}", clean_keys)))

        combined.write_parquet(out_p)
        gc.collect()
    

In [17]:
def calculate_user_item_group_features(
    day_from: int = 1250,
    day_to: int = 1301,
    num_days: int = 14,
    item_group: t.Literal["item_id", "brand_id", "category", "subcategory"] | None = None
) -> None:
    global ITEMS
    keys = ["user_id", item_group]
    clean_keys= [k for k in keys if k is not None]
    save_dir= FEATURES_DIR / "-".join(clean_keys)
    save_dir.mkdir(parents=True, exist_ok=True)

    surfaces = ["u2i", "i2i", "catalog", "search", "other"]
    actions= ["view", "click", "clickout", "like"]
    
    i_m = ITEMS.select(["item_id", item_group, "price"]) if item_group and item_group != "item_id" else ITEMS.select(["item_id", "price"])

    for target_day in tqdm(range(day_from, day_to), desc=f"U-IG: {clean_keys}"):
        day_str = str(target_day).zfill(5)
        if (save_dir / f"{day_str}.pq").exists(): continue
            
        daily_dfs= []
        for d in range(target_day - num_days, target_day):
            p = AGGS_DIR / f"{str(d).zfill(5)}.pq"
            if p.exists():
                df = pl.scan_parquet(str(p)).join(i_m.lazy(), on="item_id")
                aggs=[]
                for a in actions:
                    aggs.append(pl.col(f"num_{a}_all_subdomains").sum().alias(f"cnt_{a}"))
                    for s in surfaces:
                        aggs.append(pl.col(f"num_{a}_{s}").sum().alias(f"cnt_{a}_{s}"))
                    aggs.append(pl.col("price").filter(pl.col(f"num_{a}_all_subdomains") > 0).alias(f"p_{a}"))
                    for s in surfaces:
                        aggs.append(pl.col("price").filter(pl.col(f"num_{a}_{s}") > 0).alias(f"p_{a}_{s}"))
                
                daily_dfs.append(df.group_by(clean_keys).agg(aggs).collect())

        if not daily_dfs: continue
        
        combined = pl.concat(daily_dfs).group_by(clean_keys).agg([
            *[pl.col(f"cnt_{a}").sum().alias(_feature_name(f"num_{a}_{num_days}d", clean_keys)) for a in actions],
            *[pl.col(f"cnt_{a}_{s}").sum().alias(_feature_name(f"num_{a}_from_{s}_{num_days}d", clean_keys)) for a in actions for s in surfaces],
            *[pl.col(f"p_{a}").list.explode().median().alias(_feature_name(f"median_price_{a}_{num_days}d", clean_keys)) for a in actions],
            *[pl.col(f"p_{a}_{s}").list.explode().median().alias(_feature_name(f"median_price_{a}_from_{s}_{num_days}d", clean_keys)) for a in actions for s in surfaces]
        ])
        if "user_id" not in clean_keys or len(clean_keys) == 1:
            for p1, p2 in CONVERSION_PAIRS:
                n1, n2= _feature_name(f"num_{p1}_{num_days}d", clean_keys), _feature_name(f"num_{p2}_{num_days}d", clean_keys)
                combined = combined.with_columns((pl.col(n2)/pl.col(n1).replace(0,None)).fill_nan(0).alias(_feature_name(f"conv_{p1}_{p2}", clean_keys)))

        combined.write_parquet(save_dir / f"{day_str}.pq")
        gc.collect()

In [18]:
for item_group in [None, "item_id", "brand_id", "category"]:
    print(f"Calculating features for user_id and {item_group}")
    calculate_user_item_group_features(
        num_days=30, item_group=item_group
    )

Calculating features for user_id and None


U-IG: ['user_id']:   0%|          | 0/51 [00:00<?, ?it/s]

Calculating features for user_id and item_id


U-IG: ['user_id', 'item_id']:   0%|          | 0/51 [00:00<?, ?it/s]

Calculating features for user_id and brand_id


U-IG: ['user_id', 'brand_id']:   0%|          | 0/51 [00:00<?, ?it/s]

Calculating features for user_id and category


U-IG: ['user_id', 'category']:   0%|          | 0/51 [00:00<?, ?it/s]

In [19]:
for user_group in [None, "socdem_cluster"]:
    for item_group in ["item_id", "brand_id", "category"]:
        print(f"Calculating features for {user_group} and {item_group}")
        calculate_user_group_item_group_features(
            num_days=30, user_group=user_group, item_group=item_group
        )

Calculating features for None and item_id


UG-IG: ['item_id']:   0%|          | 0/51 [00:00<?, ?it/s]

Calculating features for None and brand_id


UG-IG: ['brand_id']:   0%|          | 0/51 [00:00<?, ?it/s]

Calculating features for None and category


...

UG-IG: ['socdem_cluster', 'item_id']:   0%|          | 0/51 [00:00<?, ?it/s]

Calculating features for socdem_cluster and brand_id


UG-IG: ['socdem_cluster', 'brand_id']:   0%|          | 0/51 [00:00<?, ?it/s]

Calculating features for socdem_cluster and category


UG-IG: ['socdem_cluster', 'category']:   0%|          | 0/51 [00:00<?, ?it/s]

Пример того, что может получиться.

In [20]:
!ls datamart/features/

brand_id  socdem_cluster-brand_id  user_id	     user_id-item_id
category  socdem_cluster-category  user_id-brand_id
item_id   socdem_cluster-item_id   user_id-category


In [21]:
pl.read_parquet("datamart/features/user_id/01251.pq").sample(10)

user_id,f_num__user_id___num_view_30d,f_num__user_id___num_click_30d,f_num__user_id___num_clickout_30d,f_num__user_id___num_like_30d,f_num__user_id___num_view_from_u2i_30d,f_num__user_id___num_view_from_i2i_30d,f_num__user_id___num_view_from_catalog_30d,f_num__user_id___num_view_from_search_30d,f_num__user_id___num_view_from_other_30d,f_num__user_id___num_click_from_u2i_30d,f_num__user_id___num_click_from_i2i_30d,f_num__user_id___num_click_from_catalog_30d,f_num__user_id___num_click_from_search_30d,f_num__user_id___num_click_from_other_30d,f_num__user_id___num_clickout_from_u2i_30d,f_num__user_id___num_clickout_from_i2i_30d,f_num__user_id___num_clickout_from_catalog_30d,f_num__user_id___num_clickout_from_search_30d,f_num__user_id___num_clickout_from_other_30d,f_num__user_id___num_like_from_u2i_30d,f_num__user_id___num_like_from_i2i_30d,f_num__user_id___num_like_from_catalog_30d,f_num__user_id___num_like_from_search_30d,f_num__user_id___num_like_from_other_30d,f_num__user_id___median_price_view_30d,f_num__user_id___median_price_click_30d,f_num__user_id___median_price_clickout_30d,f_num__user_id___median_price_like_30d,f_num__user_id___median_price_view_from_u2i_30d,f_num__user_id___median_price_view_from_i2i_30d,f_num__user_id___median_price_view_from_catalog_30d,f_num__user_id___median_price_view_from_search_30d,f_num__user_id___median_price_view_from_other_30d,f_num__user_id___median_price_click_from_u2i_30d,f_num__user_id___median_price_click_from_i2i_30d,f_num__user_id___median_price_click_from_catalog_30d,f_num__user_id___median_price_click_from_search_30d,f_num__user_id___median_price_click_from_other_30d,f_num__user_id___median_price_clickout_from_u2i_30d,f_num__user_id___median_price_clickout_from_i2i_30d,f_num__user_id___median_price_clickout_from_catalog_30d,f_num__user_id___median_price_clickout_from_search_30d,f_num__user_id___median_price_clickout_from_other_30d,f_num__user_id___median_price_like_from_u2i_30d,f_num__user_id___median_price_like_from_i2i_30d,f_num__user_id___median_price_like_from_catalog_30d,f_num__user_id___median_price_like_from_search_30d,f_num__user_id___median_price_like_from_other_30d,f_num__user_id___conv_view_click,f_num__user_id___conv_view_clickout,f_num__user_id___conv_view_like,f_num__user_id___conv_click_clickout,f_num__user_id___conv_click_like,f_num__user_id___conv_clickout_like
u64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,f32,f32,f32
78511529,8.0,2.0,0.0,0.0,6.0,2.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.225561,3.759877,null,null,4.225561,4.137584,null,null,null,0.987545,6.532209,null,null,null,null,null,null,null,null,null,null,null,null,null,0.25,0.0,0.0,0.0,0.0,null
56551825,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.288212,null,null,null,6.288212,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0.0,0.0,null,null,null
84651049,2.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.834173,null,null,null,1.834173,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0.0,0.0,null,null,null
14372415,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.089712,null,null,null,null,null,null,null,2.089712,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0.0,0.0,null,null,null
64817677,51.0,5.0,2.0,0.0,32.0,12.0,2.0,0.0,5.0,4.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.969632,1.655624,3.131552,null,1.827289,3.858843,1.45029,null,-1.76067,1.950274,null,1.644707,null,null,3.131552,null,null,null,null,null,null,null,null,null,0.098039,0.039216,0.0,0.4,0.0,0.0
88408815,3.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,

In [22]:
pl.read_parquet("datamart/features/item_id/01293.pq").sample(10)

item_id,f_num__item_id___num_view_30d,f_num__item_id___num_click_30d,f_num__item_id___num_clickout_30d,f_num__item_id___num_like_30d,f_num__item_id___num_view_from_u2i_30d,f_num__item_id___num_view_from_i2i_30d,f_num__item_id___num_view_from_catalog_30d,f_num__item_id___num_view_from_search_30d,f_num__item_id___num_view_from_other_30d,f_num__item_id___num_click_from_u2i_30d,f_num__item_id___num_click_from_i2i_30d,f_num__item_id___num_click_from_catalog_30d,f_num__item_id___num_click_from_search_30d,f_num__item_id___num_click_from_other_30d,f_num__item_id___num_clickout_from_u2i_30d,f_num__item_id___num_clickout_from_i2i_30d,f_num__item_id___num_clickout_from_catalog_30d,f_num__item_id___num_clickout_from_search_30d,f_num__item_id___num_clickout_from_other_30d,f_num__item_id___num_like_from_u2i_30d,f_num__item_id___num_like_from_i2i_30d,f_num__item_id___num_like_from_catalog_30d,f_num__item_id___num_like_from_search_30d,f_num__item_id___num_like_from_other_30d,f_num__item_id___median_price_view_30d,f_num__item_id___median_price_click_30d,f_num__item_id___median_price_clickout_30d,f_num__item_id___median_price_like_30d,f_num__item_id___conv_view_click,f_num__item_id___conv_view_click_u2i,f_num__item_id___conv_view_click_i2i,f_num__item_id___conv_view_click_catalog,f_num__item_id___conv_view_click_search,f_num__item_id___conv_view_click_other,f_num__item_id___conv_view_clickout,f_num__item_id___conv_view_clickout_u2i,f_num__item_id___conv_view_clickout_i2i,f_num__item_id___conv_view_clickout_catalog,f_num__item_id___conv_view_clickout_search,f_num__item_id___conv_view_clickout_other,f_num__item_id___conv_view_like,f_num__item_id___conv_view_like_u2i,f_num__item_id___conv_view_like_i2i,f_num__item_id___conv_view_like_catalog,f_num__item_id___conv_view_like_search,f_num__item_id___conv_view_like_other,f_num__item_id___conv_click_clickout,f_num__item_id___conv_click_clickout_u2i,f_num__item_id___conv_click_clickout_i2i,f_num__item_id___conv_click_clickout_catalog,f_num__item_id___conv_click_clickout_search,f_num__item_id___conv_click_clickout_other,f_num__item_id___conv_click_like,f_num__item_id___conv_click_like_u2i,f_num__item_id___conv_click_like_i2i,f_num__item_id___conv_click_like_catalog,f_num__item_id___conv_click_like_search,f_num__item_id___conv_click_like_other,f_num__item_id___conv_clickout_like,f_num__item_id___conv_clickout_like_u2i,f_num__item_id___conv_clickout_like_i2i,f_num__item_id___conv_clickout_like_catalog,f_num__item_id___conv_clickout_like_search,f_num__item_id___conv_clickout_like_other
str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""nfmcg_4632634""",5.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.841082,null,null,null,0.0,null,null,0.0,0.0,null,0.0,null,null,0.0,0.0,null,0.0,null,null,0.0,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""nfmcg_22877094""",415.0,15.0,0.0,0.0,311.0,8.0,23.0,13.0,60.0,11.0,1.0,0.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.25369,-0.25369,null,null,0.036145,0.03537,0.125,0.0,0.153846,0.016667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,null,0.0,0.0,0.0,0.0,0.0,null,0.0,0.0,null,null,null,null,null,null
"""nfmcg_216548""",37.0,0.0,0.0,0.0,0.0,37.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.297828,null,null,null,0.0,null,0.0,null,null,null,0.0,null,0.0,null,null,null,0.0,null,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""nfmcg_20698344""",1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.967539,null,null,null,0.0,null,0.0,null,null,null,0.0,null,0.0,null,null,null,0.0,null,0.0,null,null,null,null,null,n

In [23]:
pl.read_parquet("datamart/features/user_id-item_id/01291.pq").sample(10)

user_id,item_id,f_num__user_id_item_id___num_view_30d,f_num__user_id_item_id___num_click_30d,f_num__user_id_item_id___num_clickout_30d,f_num__user_id_item_id___num_like_30d,f_num__user_id_item_id___num_view_from_u2i_30d,f_num__user_id_item_id___num_view_from_i2i_30d,f_num__user_id_item_id___num_view_from_catalog_30d,f_num__user_id_item_id___num_view_from_search_30d,f_num__user_id_item_id___num_view_from_other_30d,f_num__user_id_item_id___num_click_from_u2i_30d,f_num__user_id_item_id___num_click_from_i2i_30d,f_num__user_id_item_id___num_click_from_catalog_30d,f_num__user_id_item_id___num_click_from_search_30d,f_num__user_id_item_id___num_click_from_other_30d,f_num__user_id_item_id___num_clickout_from_u2i_30d,f_num__user_id_item_id___num_clickout_from_i2i_30d,f_num__user_id_item_id___num_clickout_from_catalog_30d,f_num__user_id_item_id___num_clickout_from_search_30d,f_num__user_id_item_id___num_clickout_from_other_30d,f_num__user_id_item_id___num_like_from_u2i_30d,f_num__user_id_item_id___num_like_from_i2i_30d,f_num__user_id_item_id___num_like_from_catalog_30d,f_num__user_id_item_id___num_like_from_search_30d,f_num__user_id_item_id___num_like_from_other_30d,f_num__user_id_item_id___median_price_view_30d,f_num__user_id_item_id___median_price_click_30d,f_num__user_id_item_id___median_price_clickout_30d,f_num__user_id_item_id___median_price_like_30d,f_num__user_id_item_id___median_price_view_from_u2i_30d,f_num__user_id_item_id___median_price_view_from_i2i_30d,f_num__user_id_item_id___median_price_view_from_catalog_30d,f_num__user_id_item_id___median_price_view_from_search_30d,f_num__user_id_item_id___median_price_view_from_other_30d,f_num__user_id_item_id___median_price_click_from_u2i_30d,f_num__user_id_item_id___median_price_click_from_i2i_30d,f_num__user_id_item_id___median_price_click_from_catalog_30d,f_num__user_id_item_id___median_price_click_from_search_30d,f_num__user_id_item_id___median_price_click_from_other_30d,f_num__user_id_item_id___median_price_clickout_from_u2i_30d,f_num__user_id_item_id___median_price_clickout_from_i2i_30d,f_num__user_id_item_id___median_price_clickout_from_catalog_30d,f_num__user_id_item_id___median_price_clickout_from_search_30d,f_num__user_id_item_id___median_price_clickout_from_other_30d,f_num__user_id_item_id___median_price_like_from_u2i_30d,f_num__user_id_item_id___median_price_like_from_i2i_30d,f_num__user_id_item_id___median_price_like_from_catalog_30d,f_num__user_id_item_id___median_price_like_from_search_30d,f_num__user_id_item_id___median_price_like_from_other_30d
u64,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
21546260,"""nfmcg_16908441""",1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.327035,null,null,null,4.327035,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
68508585,"""nfmcg_2669352""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.901937,null,null,null,null,null,null,2.901937,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
65735407,"""nfmcg_19821679""",1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.985324,null,null,null,5.985324,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
74634419,"""nfmcg_19196260""",2.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.770755,null,null,null,null,null,2.770755,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
71136328,"""nfmcg_2669352""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.901937,null,null,null,null,null,null,2.901937,null,null,null,null,null,null,null,null,null,null,null,null,null

In [24]:
pl.read_parquet("datamart/features/socdem_cluster-brand_id/01255.pq").sample(10)

socdem_cluster,brand_id,f_num__socdem_cluster_brand_id___num_view_30d,f_num__socdem_cluster_brand_id___num_click_30d,f_num__socdem_cluster_brand_id___num_clickout_30d,f_num__socdem_cluster_brand_id___num_like_30d,f_num__socdem_cluster_brand_id___num_view_from_u2i_30d,f_num__socdem_cluster_brand_id___num_view_from_i2i_30d,f_num__socdem_cluster_brand_id___num_view_from_catalog_30d,f_num__socdem_cluster_brand_id___num_view_from_search_30d,f_num__socdem_cluster_brand_id___num_view_from_other_30d,f_num__socdem_cluster_brand_id___num_click_from_u2i_30d,f_num__socdem_cluster_brand_id___num_click_from_i2i_30d,f_num__socdem_cluster_brand_id___num_click_from_catalog_30d,f_num__socdem_cluster_brand_id___num_click_from_search_30d,f_num__socdem_cluster_brand_id___num_click_from_other_30d,f_num__socdem_cluster_brand_id___num_clickout_from_u2i_30d,f_num__socdem_cluster_brand_id___num_clickout_from_i2i_30d,f_num__socdem_cluster_brand_id___num_clickout_from_catalog_30d,f_num__socdem_cluster_brand_id___num_clickout_from_search_30d,f_num__socdem_cluster_brand_id___num_clickout_from_other_30d,f_num__socdem_cluster_brand_id___num_like_from_u2i_30d,f_num__socdem_cluster_brand_id___num_like_from_i2i_30d,f_num__socdem_cluster_brand_id___num_like_from_catalog_30d,f_num__socdem_cluster_brand_id___num_like_from_search_30d,f_num__socdem_cluster_brand_id___num_like_from_other_30d,f_num__socdem_cluster_brand_id___median_price_view_30d,f_num__socdem_cluster_brand_id___median_price_click_30d,f_num__socdem_cluster_brand_id___median_price_clickout_30d,f_num__socdem_cluster_brand_id___median_price_like_30d,f_num__socdem_cluster_brand_id___conv_view_click,f_num__socdem_cluster_brand_id___conv_view_click_u2i,f_num__socdem_cluster_brand_id___conv_view_click_i2i,f_num__socdem_cluster_brand_id___conv_view_click_catalog,f_num__socdem_cluster_brand_id___conv_view_click_search,f_num__socdem_cluster_brand_id___conv_view_click_other,f_num__socdem_cluster_brand_id___conv_view_clickout,f_num__socdem_cluster_brand_id___conv_view_clickout_u2i,f_num__socdem_cluster_brand_id___conv_view_clickout_i2i,f_num__socdem_cluster_brand_id___conv_view_clickout_catalog,f_num__socdem_cluster_brand_id___conv_view_clickout_search,f_num__socdem_cluster_brand_id___conv_view_clickout_other,f_num__socdem_cluster_brand_id___conv_view_like,f_num__socdem_cluster_brand_id___conv_view_like_u2i,f_num__socdem_cluster_brand_id___conv_view_like_i2i,f_num__socdem_cluster_brand_id___conv_view_like_catalog,f_num__socdem_cluster_brand_id___conv_view_like_search,f_num__socdem_cluster_brand_id___conv_view_like_other,f_num__socdem_cluster_brand_id___conv_click_clickout,f_num__socdem_cluster_brand_id___conv_click_clickout_u2i,f_num__socdem_cluster_brand_id___conv_click_clickout_i2i,f_num__socdem_cluster_brand_id___conv_click_clickout_catalog,f_num__socdem_cluster_brand_id___conv_click_clickout_search,f_num__socdem_cluster_brand_id___conv_click_clickout_other,f_num__socdem_cluster_brand_id___conv_click_like,f_num__socdem_cluster_brand_id___conv_click_like_u2i,f_num__socdem_cluster_brand_id___conv_click_like_i2i,f_num__socdem_cluster_brand_id___conv_click_like_catalog,f_num__socdem_cluster_brand_id___conv_click_like_search,f_num__socdem_cluster_brand_id___conv_click_like_other,f_num__socdem_cluster_brand_id___conv_clickout_like,f_num__socdem_cluster_brand_id___conv_clickout_like_u2i,f_num__socdem_cluster_brand_id___conv_clickout_like_i2i,f_num__socdem_cluster_brand_id___conv_clickout_like_catalog,f_num__socdem_cluster_brand_id___conv_clickout_like_search,f_num__socdem_cluster_brand_id___conv_clickout_like_other
u8,u64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
17,141256,224.0,2.0,0.0,0.0,9.0,4.0,14.0,0.0,197.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.205091,-1.21574,null,null,0.0089

## II. Сбор датасета. (3 балла)

Будем учить модель ранжировать показанные пользователю рекомендации в рамках дня (можно было бы выбрать и другой промежуток). Разделим подготовку обучающих данных на два этапа: сбор "скелета" (базиса) с последующим созданием датасета путем джойна фичей на базис.

### II.I Сбор датасета -> Сбор базиса (1 из 3 баллов)

Базис представляется как (`session_id`, `user_id`, `item_id`, `label`), где в качестве `session_id` используется конкатенация `user_id` и `day`, а `label` зависит от `action_type`. Напомню, что view < click < clickout < like. Сессии, целиком состоящие из view, не стоит учитывать (действительно, сложно оценить качество сортировки одинаковых элементов). Если в рамках сессии было несколько взаимодействий с айтемом, то в качестве `label` нужно взять максимальное значение.

После того, как получим предсказания модели, можно будет сгруппировать базис по `session_id` и получить структуру ([`item_id`], [`label`], [`score`]) - тогда, отсортировав по айтемы по `label` либо `score` получим список айтемов с реальной либо модельной сортировкой, а по этому уже удобно считать метрики.

Реализуйте сбор базиса. Его так же удобно сохранять "за каждый день". Добавьте логику фильтрации сессий по 99 перцентилю длины.

In [25]:
def build_basis(
    day_from: int,
    day_to: int,
    filter_99: bool = True,
    output_dir: Path = Path("output/basis/"),
) -> None:
    raw_dir= Path("datamart/raw/events/")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    action_configs= {
        "view": 1,
        "click": 2,
        "clickout": 3,
        "like": 4
    }

    processed_days=[]

    for day in tqdm(range(day_from, day_to + 1), desc="Building basis"):
        day_str = str(day).zfill(5)
        day_parts=[]
        for action_name, weight in action_configs.items():
            path= raw_dir / action_name / f"{day_str}.pq"
            if path.exists():
                part = pl.read_parquet(path).select(["user_id", "item_id"])
                part = part.with_columns(pl.lit(weight).cast(pl.Int8).alias("label"))
                day_parts.append(part)
        
        if not day_parts:
            continue

        df= pl.concat(day_parts)
        df = df.with_columns(
            (pl.col("user_id").cast(pl.String) + "_" + pl.lit(day_str)).alias("session_id")
        )
        basis_day= df.group_by(["session_id", "user_id", "item_id"]).agg(
            pl.col("label").max()
        )
        valid_sessions = (
            basis_day.group_by("session_id")
            .agg(pl.col("label").max().alias("max_l"))
            .filter(pl.col("max_l") > 1)
            .select("session_id")
        )
        basis_day= basis_day.join(valid_sessions, on="session_id")
        
        if basis_day.is_empty():
            continue
        if filter_99:
            session_lens = basis_day.group_by("session_id").len(name="sz")
            limit = session_lens.select(pl.col("sz").quantile(0.99)).item()
            valid_len_sessions = session_lens.filter(pl.col("sz") <= limit).select("session_id")
            basis_day = basis_day.join(valid_len_sessions, on="session_id")
            
        basis_day.write_parquet(output_dir / f"{day_str}.pq")
        processed_days.append(day)
        
        del df, basis_day
        gc.collect()

    print(f"Собрано дней: {len(processed_days)}")

In [26]:
build_basis(day_from=min(DAYS), day_to=max(DAYS), filter_99=True)

Building basis:   0%|          | 0/51 [00:00<?, ?it/s]

Собрано дней: 51


In [27]:
basis = pl.read_parquet("output/basis/")
print(basis.shape)
basis.sample(10)

(1140313, 4)


session_id,user_id,item_id,label
str,u64,str,i8
"""19264857_01300""",19264857,"""nfmcg_25465317""",3
"""75635683_01289""",75635683,"""nfmcg_1828041""",1
"""51845016_01296""",51845016,"""nfmcg_18804131""",1
"""78399103_01292""",78399103,"""nfmcg_11729259""",1
"""80385047_01294""",80385047,"""nfmcg_14457630""",1
"""87127290_01295""",87127290,"""nfmcg_8571996""",1
"""70069856_01294""",70069856,"""nfmcg_17133555""",1
"""40095048_01295""",40095048,"""nfmcg_5830295""",1
"""45285004_01294""",45285004,"""nfmcg_15044500""",1


In [28]:
for col in basis.columns:
    print(f"{col}: {basis[col].n_unique()}")

session_id: 38231
user_id: 15877
item_id: 19968
label: 4


In [29]:
basis["label"].value_counts().sort("label")

label,count
i8,u32
1,1037031
2,86361
3,15092
4,1829


In [30]:
basis.group_by("session_id").agg(pl.len())["len"].describe()

statistic,value
str,f64
"""count""",38231.0
"""null_count""",0.0
"""mean""",29.826921
"""std""",31.040807
"""min""",1.0
"""25%""",8.0
"""50%""",20.0
"""75%""",40.0
"""max""",207.0


### II.II Сбор датасета -> Сбор датасета с фичами (2 из 3 баллов)

Чтобы создать датасет, достаточно приджойнить к базису фичи. Обратите внимание, что фичи должны быть собраны за предыдущий день, чтобы избежать ликов. То есть, если мы работаем с базисом на 1300 день, то фичи для него необходимо брать из 1299 дня. Помимо числовых, можно также добавить категориальные фичи.

Реализуйте необходимую логику. Добавьте возможность читать список фичей, которые необходимо приджойнить, из файла.

In [31]:
def join_features(
    df: pl.LazyFrame,
    day: int,
    users: pl.LazyFrame,
    items: pl.LazyFrame,
    features_dir: Path = Path("datamart/features/"),
    features_to_use: list[str] | None = None,
) -> pl.LazyFrame:
    feature_day_str= str(day - 1).zfill(5)
    
    if features_to_use is None:
        features_to_use = [d.name for d in features_dir.iterdir() if d.is_dir()]
    df= df.join(users.select(["user_id", "socdem_cluster", "region"]), on="user_id", how="left")
    df= df.join(items.select(["item_id", "brand_id", "category", "subcategory"]), on="item_id", how="left")

    for feature_group in features_to_use:
        join_keys= feature_group.split("-")
        
        feat_path= features_dir / feature_group / f"{feature_day_str}.pq"
        
        if feat_path.exists():
            f_df = pl.scan_parquet(feat_path)
            df = df.join(f_df, on=join_keys, how="left")
        else:
            print(f"Warning: Feature file {feat_path} not found for day {day}")

    return df


def build_dataset(
    day_from: int,
    day_to: int,
    basis_dir: Path = Path("output/basis/"),
    output_dir: Path = Path("output/dataset/"),
    features_to_use_filepath: Path | None = None,
    users: pl.LazyFrame | None = None,
    items: pl.LazyFrame | None = None,
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    features_to_use = None
    if features_to_use_filepath and features_to_use_filepath.exists():
        with open(features_to_use_filepath, "r") as f:
            features_to_use = [line.strip() for line in f if line.strip()]

    if users is None: users = USERS.lazy()
    if items is None: items = ITEMS.lazy()

    processed_count = 0
    for day in tqdm(range(day_from, day_to + 1), desc="Building dataset"):
        day_str = str(day).zfill(5)
        basis_path = basis_dir / f"{day_str}.pq"
        
        if not basis_path.exists():
            continue
        lf= pl.scan_parquet(basis_path)
                
        lf = join_features(
            lf, 
            day=day, 
            users=users, 
            items=items, 
            features_to_use=features_to_use
        )
        lf = lf.with_columns([
            pl.col("^f_num.*$").fill_null(0)
        ])
        lf.collect().write_parquet(output_dir / f"{day_str}.pq")
        processed_count +=1
        gc.collect()

    print(f"Dataset build complete. Days processed: {processed_count}")


In [32]:
days_int = [int(d) for d in DAYS]
build_dataset(
    day_from=min(days_int), 
    day_to=max(days_int), 
    users=USERS.lazy(), 
    items=ITEMS.lazy()
)

Building dataset:   0%|          | 0/51 [00:00<?, ?it/s]

Dataset build complete. Days processed: 51


In [33]:
pl.read_parquet("output/dataset/01299.pq").sample(10)

session_id,user_id,item_id,label,socdem_cluster,region,brand_id,category,subcategory,f_num__socdem_cluster_item_id___num_view_30d,f_num__socdem_cluster_item_id___num_click_30d,f_num__socdem_cluster_item_id___num_clickout_30d,f_num__socdem_cluster_item_id___num_like_30d,f_num__socdem_cluster_item_id___num_view_from_u2i_30d,f_num__socdem_cluster_item_id___num_view_from_i2i_30d,f_num__socdem_cluster_item_id___num_view_from_catalog_30d,f_num__socdem_cluster_item_id___num_view_from_search_30d,f_num__socdem_cluster_item_id___num_view_from_other_30d,f_num__socdem_cluster_item_id___num_click_from_u2i_30d,f_num__socdem_cluster_item_id___num_click_from_i2i_30d,f_num__socdem_cluster_item_id___num_click_from_catalog_30d,f_num__socdem_cluster_item_id___num_click_from_search_30d,f_num__socdem_cluster_item_id___num_click_from_other_30d,f_num__socdem_cluster_item_id___num_clickout_from_u2i_30d,f_num__socdem_cluster_item_id___num_clickout_from_i2i_30d,f_num__socdem_cluster_item_id___num_clickout_from_catalog_30d,f_num__socdem_cluster_item_id___num_clickout_from_search_30d,f_num__socdem_cluster_item_id___num_clickout_from_other_30d,f_num__socdem_cluster_item_id___num_like_from_u2i_30d,f_num__socdem_cluster_item_id___num_like_from_i2i_30d,f_num__socdem_cluster_item_id___num_like_from_catalog_30d,f_num__socdem_cluster_item_id___num_like_from_search_30d,f_num__socdem_cluster_item_id___num_like_from_other_30d,f_num__socdem_cluster_item_id___median_price_view_30d,f_num__socdem_cluster_item_id___median_price_click_30d,f_num__socdem_cluster_item_id___median_price_clickout_30d,f_num__socdem_cluster_item_id___median_price_like_30d,…,f_num__socdem_cluster_category___median_price_like_30d,f_num__socdem_cluster_category___conv_view_click,f_num__socdem_cluster_category___conv_view_click_u2i,f_num__socdem_cluster_category___conv_view_click_i2i,f_num__socdem_cluster_category___conv_view_click_catalog,f_num__socdem_cluster_category___conv_view_click_search,f_num__socdem_cluster_category___conv_view_click_other,f_num__socdem_cluster_category___conv_view_clickout,f_num__socdem_cluster_category___conv_view_clickout_u2i,f_num__socdem_cluster_category___conv_view_clickout_i2i,f_num__socdem_cluster_category___conv_view_clickout_catalog,f_num__socdem_cluster_category___conv_view_clickout_search,f_num__socdem_cluster_category___conv_view_clickout_other,f_num__socdem_cluster_category___conv_view_like,f_num__socdem_cluster_category___conv_view_like_u2i,f_num__socdem_cluster_category___conv_view_like_i2i,f_num__socdem_cluster_category___conv_view_like_catalog,f_num__socdem_cluster_category___conv_view_like_search,f_num__socdem_cluster_category___conv_view_like_other,f_num__socdem_cluster_category___conv_click_clickout,f_num__socdem_cluster_category___conv_click_clickout_u2i,f_num__socdem_cluster_category___conv_click_clickout_i2i,f_num__socdem_cluster_category___conv_click_clickout_catalog,f_num__socdem_cluster_category___conv_click_clickout_search,f_num__socdem_cluster_category___conv_click_clickout_other,f_num__socdem_cluster_category___conv_click_like,f_num__socdem_cluster_category___conv_click_like_u2i,f_num__socdem_cluster_category___conv_click_like_i2i,f_num__socdem_cluster_category___conv_click_like_catalog,f_num__socdem_cluster_category___conv_click_like_search,f_num__socdem_cluster_category___conv_click_like_other,f_num__socdem_cluster_category___conv_clickout_like,f_num__socdem_cluster_category___conv_clickout_like_u2i,f_num__socdem_cluster_category___conv_clickout_like_i2i,f_num__socdem_cluster_category___conv_clickout_like_catalog,f_num__socdem_cluster_category___conv_clickout_like_search,f_num__socdem_cluster_category___conv_clickout_like_other
str,u64,str,i8,u8,u8,u64,str,str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f64,…,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""61447375_01299""",614

## III Обучение ранжирования (4 балла)

### III.I. Подготовка выборок для обучения/валидации/теста и реализация метрик (1 из 4 баллов)

Полезно будет также написать функцию, считывающую с диска и возвращающую train, val, train_val, и test части датасета. Диапазон будем задавать через дни.

In [34]:
def read_dataset(
    day_from: int,
    n_train_days: int,
    n_val_days: int,
    n_test_days: int,
    dataset_dir: Path = Path("output/dataset/"),
) -> tuple[pl.LazyFrame, pl.LazyFrame, pl.LazyFrame, pl.LazyFrame]:
    train_end = day_from+ n_train_days
    val_end= train_end+n_val_days
    test_end= val_end+ n_test_days
    
    def scan_days(start: int, end: int) -> pl.LazyFrame:
        paths =[]
        for day in range(start, end):
            path= dataset_dir/f"{str(day).zfill(5)}.pq"
            if path.exists():
                paths.append(str(path))
        
        if not paths:
            print(f"Warning: No files found in range [{start}, {end})")
            return pl.LazyFrame()
            
        return pl.scan_parquet(paths)

    train_lf= scan_days(day_from, train_end)
    val_lf = scan_days(train_end, val_end)
    train_val_lf= scan_days(day_from, val_end)
    
    test_lf= scan_days(val_end, test_end)
    
    return train_lf, val_lf, train_val_lf, test_lf

In [35]:
train_df, val_df, train_val_df, test_df = read_dataset(
    day_from=1265,
    n_train_days=14,
    n_val_days=1,
    n_test_days=1,
    dataset_dir=Path("output/dataset/")
)
train_df= train_df.collect()
val_df= val_df.collect()
train_val_df= train_val_df.collect()
test_df= test_df.collect()

In [36]:
train_val_df.shape

(129378, 591)

In [37]:
test_df.shape

(8831, 591)

Реализуйте логику расчета метрик по структуре ([`item_id`], [`label`], [`score`]) - можете формировать эту структуру также внутри функции расчета метрик, а можете вне. В качестве метрики обязательно использовать NDCG@k. В выборе остальных метрик вы свободны. Полезно может быть считать метрики в разрезе по label - например, сколько айтемов с label=2 попали в топ-10 рекомендаций.

Посчитайте метрики в A/A-сеттинге.

In [38]:
class AtKMetric(ABC):
    def __init__(self, k: int):
        self.k= k

    @property
    @abstractmethod
    def name(self) -> str:
        raise NotImplementedError

    @property
    def full_name(self) -> str:
        return f"{self.name}@{self.k}"

    @abstractmethod
    def __call__(self, *, labels_col: str = "labels", targets_col: str = "targets") -> pl.Expr:
        raise NotImplementedError


class NdcgAtK(AtKMetric):
    @property
    def name(self) -> str:
        return "ndcg"
    def __call__(self, *, preds_col: str = "preds", targets_col: str = "targets") -> pl.Expr:
        return (
            pl.struct([preds_col, targets_col]).map_elements(
                lambda x: self._calculate_ndcg(x[preds_col], x[targets_col]),
                return_dtype=pl.Float64
            ).alias(self.full_name)
        )

    def _calculate_ndcg(self, preds, targets):
        target_map= {t['item_id']: t['label'] for t in targets}
        predicted_labels= [target_map.get(p['item_id'], 0) for p in preds[:self.k]]
        ideal_labels= sorted([t['label'] for t in targets], reverse=True)[:self.k]
        
        def dcg(labels):
            return sum((2**l-1)/np.log2(i+2) for i, l in enumerate(labels))
        
        actual_dcg= dcg(predicted_labels)
        ideal_dcg= dcg(ideal_labels)
        
        return actual_dcg/ideal_dcg if ideal_dcg > 0 else 0.0

class RecallAtK(AtKMetric):
    @property
    def name(self) -> str:
        return "recall"

    def __call__(self, *, preds_col: str = "preds", targets_col: str = "targets") -> pl.Expr:
        return (
            pl.struct([preds_col, targets_col]).map_elements(
                lambda x: self._calculate_recall(x[preds_col], x[targets_col]),
                return_dtype=pl.Float64
            ).alias(self.full_name)
        )

    def _calculate_recall(self, preds, targets):
        relevant_items = {t['item_id'] for t in targets if t['label'] > 1}
        if not relevant_items: return 0.0
        
        top_k_items= {p['item_id'] for p in preds[:self.k]}
        hits = len(relevant_items.intersection(top_k_items))
        return hits / len(relevant_items)
        
def evaluate_ranker(
    df: pl.DataFrame,
    ks: list[int] = [1,  5, 10, 20, 50],
    preds_col: str = "preds",
    targets_col: str = "targets",
) -> pl.DataFrame:
    metrics = []
    for k in ks:
        metrics.append(NdcgAtK(k))
        metrics.append(RecallAtK(k))
    exprs = [m(preds_col=preds_col, targets_col=targets_col) for m in metrics]
    results = df.select(exprs).mean()
    return results.to_dict(as_series=False)

In [39]:
metrics_best = evaluate_ranker(
    test_df.with_columns(score=pl.col("label"))
    .group_by("session_id").agg(
        [
            pl.struct("item_id").sort_by("score", descending=True).alias("preds"),
            pl.struct("item_id", "label").sort_by("label", descending=True).alias("targets"),
        ]
    )
)
metrics_worst = evaluate_ranker(
    test_df.with_columns(score=pl.col("label"))
    .group_by("session_id").agg(
        [
            pl.struct("item_id").sort_by("score", descending=False).alias("preds"),
            pl.struct("item_id", "label").sort_by("label", descending=True).alias("targets"),
        ]
    )
)
RESULTS = pd.concat([
    pd.DataFrame(metrics_best, index=["best"]),
    pd.DataFrame(metrics_worst, index=["worst"]),
])
RESULTS.style.format(precision=5).background_gradient(cmap="Blues")

,ndcg@1,recall@1,ndcg@5,recall@5,ndcg@10,recall@10,ndcg@20,recall@20,ndcg@50,recall@50
best,1.00000,0.65702,1.00000,0.98203,1.00000,0.99868,1.00000,1.00000,1.00000,1.00000
worst,0.37279,0.11144,0.55656,0.29904,0.65780,0.54453,0.72182,0.75882,0.75892,0.94118


### III.II. Реализация TopPopular бейзлайна (1 из 4 баллов)

Реализуйте любой бейзлайн (бейзлайны) на своё усмотрение. Посчитайте метрики. Не забудьте про консистентность: учимся на train - оцениваем на val; учимся на train+val - оцениваем на test.

Важно: используйте `sample(fraction=1.0, shuffle=True)` при группировке по сессии для расчета метрик, чтобы в случае одинаковых скоров автоматом не проставлялся скор из корркетно отсортированной последовтаельности айтемов!

In [40]:
baseline_score_col = "f_num__item_id___num_click_30d"
baseline_data= (
    test_df
    .select(["session_id", "item_id", "label", baseline_score_col])
    .sample(fraction=1.0, shuffle=True)
    .group_by("session_id")
    .agg([
        pl.struct("item_id")
          .sort_by(baseline_score_col, descending=True)
          .alias("preds"),
        pl.struct(["item_id", "label"])
          .sort_by("label", descending=True)
          .alias("targets"),
    ])
)

In [41]:
baseline = evaluate_ranker(
    baseline_data,
    ks=[1, 5, 10, 20, 50],
    preds_col="preds",
    targets_col="targets"
)

RESULTS = pd.concat([
    RESULTS,
    pd.DataFrame(baseline, index=["baseline"]),
])
RESULTS.style.format(precision=5).background_gradient(cmap="Blues")

,ndcg@1,recall@1,ndcg@5,recall@5,ndcg@10,recall@10,ndcg@20,recall@20,ndcg@50,recall@50
best,1.00000,0.65702,1.00000,0.98203,1.00000,0.99868,1.00000,1.00000,1.00000,1.00000
worst,0.37279,0.11144,0.55656,0.29904,0.65780,0.54453,0.72182,0.75882,0.75892,0.94118
baseline,0.51731,0.23740,0.71398,0.63020,0.78853,0.81830,0.82876,0.93322,0.84713,0.99025


In [42]:
test_df['user_id'].n_unique()

545

### III.III. Реализация обучения градиентного бустинга (2 из 4 баллов)

Обучите ранкер на train части датасета. В качестве модели можете использовать любую из [Catboost](https://catboost.ai), [LGBM](https://lightgbm.readthedocs.io/en/latest/pythonapi/lightgbm.Booster.html), [XGBoost](https://xgboost.readthedocs.io/en/stable/). Обучать можно как Ranker, так и Classifier, так и Regressor. Поэкспериментируйте. Посчитайте метрики на val части и подберите гиперпараметры (можете взять разные временные срезы, чтобы не заоверфиттиться под один).

Обучите итоговую модель на train + val, замерьте качество на test и сравните с бейзлайном.

Sanity check. Обратите внимание, что если вы считаете бейзлайн по фиче из датасета, то фича, по которой вы считаете бейзлайн, обязательно должна присутствовать как фича для ранкера. Если ранкер при использовании этой фичи показывает результаты хуже, чем бейзлайн, то что-то с вашим ранкером не так.

In [43]:
features= [col for col in train_df.columns if col.startswith("f_")]
categorical_features = [col for col in features if col.startswith("f_cat__")]

In [44]:
def train_evaluate_model(params, train_df, val_df, features, categorical_features):
    train_df= train_df.sort("session_id")
    val_df= val_df.sort("session_id")
    
    train_pool = Pool(
        data=train_df.select(features).to_pandas(),
        label=train_df["label"].to_pandas(),
        group_id=train_df["session_id"].to_pandas(),
        cat_features=categorical_features
    )
    
    val_pool = Pool(
        data=val_df.select(features).to_pandas(),
        label=val_df["label"].to_pandas(),
        group_id=val_df["session_id"].to_pandas(),
        cat_features=categorical_features
    )
    
    model = CatBoostRanker(**params, loss_function='YetiRank', random_seed=42, verbose=False)
    model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50)
    
    preds=model.predict(val_pool)
    val_with_preds = val_df.with_columns(score=pl.Series(preds))
    
    eval_data = (
        val_with_preds.sample(fraction=1.0, shuffle=True)
        .group_by("session_id")
        .agg([
            pl.struct("item_id").sort_by("score", descending=True).alias("preds"),
            pl.struct(["item_id", "label"]).sort_by("label", descending=True).alias("targets"),
        ])
    )
    
    metrics = evaluate_ranker(eval_data, ks=[10])
    return metrics["ndcg@10"][0]


def objective(trial):
    params = {
        "iterations": 500,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
    }
    return train_evaluate_model(params, train_df, val_df, features, categorical_features)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

print(f"\nBest ndcg@10: {study.best_value:.5f}")


[I 2026-04-15 16:21:34,782] A new study created in memory with name: no-name-114ae970-d96b-43dc-bb26-ab255011614b
[I 2026-04-15 16:21:53,256] Trial 0 finished with value: 0.7738905712773607 and parameters: {'learning_rate': 0.11761409110092379, 'depth': 8, 'l2_leaf_reg': 1.4005914204971472}. Best is trial 0 with value: 0.7738905712773607.
[I 2026-04-15 16:22:16,336] Trial 1 finished with value: 0.7727344460937038 and parameters: {'learning_rate': 0.032512163348769794, 'depth': 5, 'l2_leaf_reg': 9.927450433827797}. Best is trial 0 with value: 0.7738905712773607.
[I 2026-04-15 16:23:20,899] Trial 2 finished with value: 0.7780044183916448 and parameters: {'learning_rate': 0.06536061422753207, 'depth': 8, 'l2_leaf_reg': 8.70994748922043}. Best is trial 2 with value: 0.7780044183916448.
[I 2026-04-15 16:23:59,992] Trial 3 finished with value: 0.7719691826371763 and parameters: {'learning_rate': 0.010785259762187452, 'depth': 5, 'l2_leaf_reg': 4.669509735416677}. Best is trial 2 with value: 


Best ndcg@10: 0.77800


In [45]:
best_params= study.best_params
best_params["iterations"]= 1000 

full_train_df = pl.concat([train_df, val_df]).sort("session_id")
test_df = test_df.sort("session_id")

train_val_pool= Pool(
    data=full_train_df.select(features).to_pandas(),
    label=full_train_df["label"].to_pandas(),
    group_id=full_train_df["session_id"].to_pandas(),
    cat_features=categorical_features
)

test_pool= Pool(
    data=test_df.select(features).to_pandas(),
    cat_features=categorical_features
)

final_model = CatBoostRanker(**best_params, loss_function='YetiRank', random_seed=42, verbose=100)
final_model.fit(train_val_pool)

test_preds=final_model.predict(test_pool)
test_df_with_preds= test_df.with_columns(score=pl.Series(test_preds))

test_eval_data = (
    test_df_with_preds.sample(fraction=1.0, shuffle=True)
    .group_by("session_id")
    .agg([
        pl.struct("item_id").sort_by("score", descending=True).alias("preds"),
        pl.struct(["item_id", "label"]).sort_by("label", descending=True).alias("targets"),
    ])
)

ranker_results= evaluate_ranker(test_eval_data, ks=[1, 5, 10, 20, 50])

0:	total: 357ms	remaining: 5m 56s
100:	total: 28.4s	remaining: 4m 12s
200:	total: 53.8s	remaining: 3m 33s
300:	total: 1m 18s	remaining: 3m 3s
400:	total: 1m 43s	remaining: 2m 34s
500:	total: 2m 8s	remaining: 2m 7s
600:	total: 2m 33s	remaining: 1m 41s
700:	total: 2m 58s	remaining: 1m 16s
800:	total: 3m 23s	remaining: 50.5s
900:	total: 3m 48s	remaining: 25.1s
999:	total: 4m 13s	remaining: 0us


In [48]:
RESULTS = pd.concat([
    RESULTS,
    pd.DataFrame(ranker_results, index=["ranker"]),
])
RESULTS.style.format(precision=5).background_gradient(cmap="Blues")

,ndcg@1,recall@1,ndcg@5,recall@5,ndcg@10,recall@10,ndcg@20,recall@20,ndcg@50,recall@50
best,1.00000,0.65702,1.00000,0.98203,1.00000,0.99868,1.00000,1.00000,1.00000,1.00000
worst,0.37279,0.11144,0.55656,0.29904,0.65780,0.54453,0.72182,0.75882,0.75892,0.94118
baseline,0.51731,0.23740,0.71398,0.63020,0.78853,0.81830,0.82876,0.93322,0.84713,0.99025
ranker,0.55872,0.27826,0.73671,0.65990,0.80948,0.84776,0.84396,0.93121,0.86140,0.98879


Опишите полученный результат - получилось ли обогнать бустингом baseline? Почему?

Бустингу удалось обогнать бейзлай почти по всем метрикам, за исключением recall@20 и recall@50, но результаты по ним практически идентичны. Задача ранкера была в том, чтобы поднять нужные товары из списка выше, и он прекрасно справился, если судить по ndcg@1 и ndcg@10 - модель хорошо ранжирует товары и ставит релефантные на самое первое место.

Бустинг оказался лучше по ряду причин:
1 - Персонализация. Бейзлайн популярное одинаков для всех. А бустинг учитывает взаимодействия пользователья с товаром, категорией.
2 - Бустинг является нелинейной моделью, поэтому лучше улавливает сложные зависимости.
3 - Бустинг учитывает socdem-признаки.